# How to use the Matcher framework?

The `Matcher` framework provides a high-level interface for computing correspondences between shapes. It encapsulates the full functional map pipeline into a single, configurable class.

In [ ]:
import gsops.backend as gs

from geomfum.dataset import NotebooksDataset
from geomfum.matcher import (
    FeatureMatcher,
    FeatureMatcherConfig,
    FunctionalMapMatcher,
    MatcherConfig,
    PreciseMatcher,
    QuickMatcher,
)
from geomfum.shape import TriangleMesh

[Load meshes](00_load_mesh_from_file.ipynb).

In [ ]:
dataset = NotebooksDataset()

mesh_a = TriangleMesh.from_file(dataset.get_filename("faust-00"))
mesh_b = TriangleMesh.from_file(dataset.get_filename("faust-04"))

mesh_a.n_vertices, mesh_b.n_vertices

INFO:root:Downloading 'C:\Users\giuli\.geomfum\data\faust-00.off' from https://raw.githubusercontent.com/JM-data/PyFuncMap/4bde4484c3e93bff925a6a82da29fa79d6862f4b/FAUST_shapes_off//tr_reg_000.off to 'C:\Users\giuli\.geomfum\data'.
INFO:root:Downloading 'C:\Users\giuli\.geomfum\data\faust-04.off' from https://raw.githubusercontent.com/JM-data/PyFuncMap/4bde4484c3e93bff925a6a82da29fa79d6862f4b/FAUST_shapes_off//tr_reg_004.off to 'C:\Users\giuli\.geomfum\data'.


(6890, 6890)

## Feature Matcher

The simplest way to match shapes is computing features and performing nearest neighbor search, this routine is made by the Feature matcher.



In [ ]:
# Basic usage with defaults
matcher = FeatureMatcher()
result = matcher(mesh_a, mesh_b)
p2p21 = result.p2p21  # Maps each vertex in B to a vertex in A

print(f"P2P21 shape: {p2p21.shape}")

P2P21 shape: (6890,)


The result contains:
- `p2p21`: point-to-point correspondence from B to A (for each vertex in B, gives corresponding vertex in A)
- `descr_a`, `descr_b`: computed descriptors
- `fmap12`: functional map matrix from A to B (None for FeatureMatcher)
- `refined_fmap12`: refined functional map (None for FeatureMatcher)

In [ ]:
result.descr_a.shape, result.descr_b.shape

((400, 6890), (400, 6890))

## Functional Map Matcher

For more robust matching, use `FunctionalMapMatcher` which optimizes a functional map.

In [ ]:
matcher = FunctionalMapMatcher()
result = matcher(mesh_a, mesh_b)

print(f"P2P21 shape: {result.p2p21.shape}")  # B -> A
print(f"Fmap12 shape: {result.fmap12.shape}")  # A -> B
print(f"Refined Fmap12 shape: {result.refined_fmap12.shape}")

P2P21 shape: (6890,)
Fmap12 shape: (30, 30)
Refined Fmap12 shape: (80, 80)


## Using landmarks

[Set landmarks](./06_landmarks.ipynb) on both shapes for better matching.

In [ ]:
mesh_a.set_landmarks(gs.array([412, 5891, 6593, 3323, 2119]))
mesh_b.set_landmarks(gs.array([412, 5891, 6593, 3323, 2119]))

Use landmarks by adding `LandmarkWaveKernelSignature` to the descriptors list.

In [ ]:
from geomfum.descriptor.pipeline import ArangeSubsampler
from geomfum.descriptor.spectral import LandmarkWaveKernelSignature, WaveKernelSignature

config = MatcherConfig(
    descriptors=[
        WaveKernelSignature.from_registry(n_domain=200),
        LandmarkWaveKernelSignature.from_registry(n_domain=200),
    ],
    subsamplers=[ArangeSubsampler(subsample_step=5)],
)

matcher = FunctionalMapMatcher(config=config)
result = matcher(mesh_a, mesh_b)

result.p2p21.shape

(6890,)

## Preset matchers

Several preset matchers are available for common use cases.

### QuickMatcher

Fast matching with reduced settings (smaller spectrum, fewer refinement iterations).

In [ ]:
quick_matcher = QuickMatcher()

result = quick_matcher(mesh_a, mesh_b)

result.p2p21.shape

(6890,)

### PreciseMatcher

High-quality matching with larger settings.

In [ ]:
precise_matcher = PreciseMatcher(use_landmarks=True)

result = precise_matcher(mesh_a, mesh_b)

result.p2p21.shape

(6890,)

## Custom configuration

Use `MatcherConfig` to fully customize the matching pipeline.

In [ ]:
from geomfum.descriptor.spectral import HeatKernelSignature
from geomfum.refine import IcpRefiner, ZoomOut

config = MatcherConfig(
    spectrum_size=100,  # Number of eigenfunctions to compute
    fmap_size=20,  # Size of functional map matrix
    descriptors=[  # Custom descriptors
        HeatKernelSignature.from_registry(n_domain=100),
        WaveKernelSignature.from_registry(n_domain=200),
    ],
    subsamplers=[ArangeSubsampler(subsample_step=10)],
    sdp_weight=1.0,  # Weight for descriptor preservation
    lb_weight=1e-2,  # Weight for LB commutativity
    mult_weight=1e-1,  # Weight for multiplication commutativity
    orient_weight=0.0,  # Weight for orientation commutativity
    refiners=[  # Custom refinement pipeline
        IcpRefiner(nit=5),
        ZoomOut(nit=4, step=5),
    ],
)

custom_matcher = FunctionalMapMatcher(config=config)
result = custom_matcher(mesh_a, mesh_b)

result.p2p21.shape

(6890,)

## Custom refiners

You can specify any sequence of [refiners](./15_refine_functional_map.ipynb) in the config.

In [ ]:
from geomfum.refine import OrthogonalRefiner

# Use orthogonal projection followed by ICP
config = MatcherConfig(
    refiners=[
        OrthogonalRefiner(),
        IcpRefiner(nit=10),
    ]
)

matcher = FunctionalMapMatcher(config=config)
result = matcher(mesh_a, mesh_b)

result.p2p21.shape

(6890,)

In [ ]:
# Disable refinement entirely
config = MatcherConfig(refiners=[])

matcher = FunctionalMapMatcher(config=config)
result = matcher(mesh_a, mesh_b)

result.refined_fmap12  # Should be None

## Custom FeatureMatcher

You can also customize the `FeatureMatcher` with custom descriptors.

In [ ]:
config = FeatureMatcherConfig(
    spectrum_size=100,
    descriptors=[
        HeatKernelSignature.from_registry(n_domain=50),
        WaveKernelSignature.from_registry(n_domain=100),
    ],
)

matcher = FeatureMatcher(config=config)
result = matcher(mesh_a, mesh_b)

result.p2p21.shape

(6890,)

## Further reading

* [How to compute a functional map?](./07_functional_map.ipynb)

* [How to refine a functional map?](./15_refine_functional_map.ipynb)

* [How to create a descriptor pipeline?](./04_descriptor_pipeline.ipynb)

* [How to set landmarks?](./06_landmarks.ipynb)